In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# OASIS U-NET RECONSTRUCTION CI 
# ══════════════════════════════════════════════════════════════════════════════

import os, gc, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

# ─────────────────────────────────────────────────────────────────────────────
# BASE PATHS (ADJUSTED TO YOUR PROJECT)
# ─────────────────────────────────────────────────────────────────────────────
BASE = "/home/jupyter/notebooks/reid_clean"

OASIS_UNET_ROOT = Path(f"{BASE}/full_pipeline/data/oasis/processed/unet")
OASIS_FUNC_DIR  = OASIS_UNET_ROOT / "functionality_test"
OASIS_REID_ROOT = Path(f"{BASE}/full_pipeline/models/oasis/reid")

SAVE_DIR = Path(f"{BASE}/full_pipeline/results/oasis/recon")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BOOTSTRAP_B = 2000
CI_LOW = 2.5
CI_HIGH = 97.5
RANDOM_SEED = 42

MODEL_DISPLAY = {
    "resnet18": "ResNet-18",
    "densenet121": "DenseNet-121",
}

# ─────────────────────────────────────────────────────────────────────────────
# SSIM + PSNR
# ─────────────────────────────────────────────────────────────────────────────
def _gaussian_window(window_size=11, sigma=1.5, channels=1, device="cpu"):
    coords = torch.arange(window_size, device=device).float() - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w = (g[:, None] * g[None, :]).unsqueeze(0).unsqueeze(0)
    return w.repeat(channels, 1, 1, 1)

def ssim_torch(x, y, data_range=1.0):
    x, y = x.float(), y.float()
    ch = x.shape[1]
    w = _gaussian_window(11, 1.5, ch, x.device)

    mu_x = F.conv2d(x, w, padding=5, groups=ch)
    mu_y = F.conv2d(y, w, padding=5, groups=ch)

    sigma_x = F.conv2d(x*x, w, padding=5, groups=ch) - mu_x**2
    sigma_y = F.conv2d(y*y, w, padding=5, groups=ch) - mu_y**2
    sigma_xy = F.conv2d(x*y, w, padding=5, groups=ch) - mu_x*mu_y

    C1 = (0.01 * data_range)**2
    C2 = (0.03 * data_range)**2

    ssim = ((2*mu_x*mu_y + C1)*(2*sigma_xy + C2)) / \
           ((mu_x**2 + mu_y**2 + C1)*(sigma_x + sigma_y + C2))

    return ssim.mean(dim=(1,2,3))

def psnr_torch(x, y):
    mse = F.mse_loss(x, y, reduction="none").mean(dim=(1,2,3))
    return 10 * torch.log10(1.0 / (mse + 1e-8))

# ─────────────────────────────────────────────────────────────────────────────
# Bootstrap
# ─────────────────────────────────────────────────────────────────────────────
def bootstrap_ci(vals):
    vals = np.asarray(vals)
    rng = np.random.default_rng(RANDOM_SEED)
    n = len(vals)

    samples = []
    for _ in range(BOOTSTRAP_B):
        idx = rng.choice(n, n, replace=True)
        samples.append(np.mean(vals[idx]))

    samples = np.array(samples)
    return (
        float(vals.mean()),
        float(np.percentile(samples, CI_LOW)),
        float(np.percentile(samples, CI_HIGH))
    )

# ─────────────────────────────────────────────────────────────────────────────
# Re-ID Dataset + Model
# ─────────────────────────────────────────────────────────────────────────────
class AdaptiveNormDataset(Dataset):
    def __init__(self, imgs, labels, mean, std):
        self.imgs = imgs
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx]
        img = (img - self.mean) / (self.std + 1e-8)
        return torch.tensor(img).unsqueeze(0).float(), self.labels[idx]

def get_model(arch):
    if arch == "resnet18":
        model = models.resnet18(weights=None)
        model.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        model.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(512,347))
    else:
        model = models.densenet121(weights=None)
        model.features[0] = nn.Conv2d(1,64,7,2,3,bias=False)
        model.classifier = nn.Sequential(nn.Dropout(0.0), nn.Linear(1024,347))
    return model.to(DEVICE)

def load_reid_model(arch):
    fname = f"{arch}_Pretrained_Raw_best.pth"
    path = OASIS_REID_ROOT / arch / fname

    model = get_model(arch)
    ckpt = torch.load(path, map_location=DEVICE)

    sd = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    model.load_state_dict({k.replace("module.",""):v for k,v in sd.items()})

    model.eval()
    print(f"✓ Loaded {arch}")
    return model

# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
x_raw = torch.load(OASIS_FUNC_DIR / "X_func_raw.pt", map_location="cpu")
labels = torch.load(OASIS_FUNC_DIR / "labels_reid.pt", map_location="cpu").numpy()

if x_raw.ndim == 3:
    x_raw = x_raw.unsqueeze(1)

print("Loaded OASIS:")
print("  raw:", x_raw.shape)
print("  labels:", labels.shape)

# ─────────────────────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────────────────────
rows = []

for condition, file in [
    ("gs0", "X_func_recon_p0.pt"),
    ("gs50", "X_func_recon_p50.pt"),
]:
    print(f"\n=== {condition.upper()} ===")

    recon = torch.load(OASIS_FUNC_DIR / file, map_location="cpu")
    if recon.ndim == 3:
        recon = recon.unsqueeze(1)

    # ── INTENSITY ─────────────────────────────────────────
    ssim_vals = ssim_torch(recon, x_raw).numpy()
    psnr_vals = psnr_torch(recon, x_raw).numpy()

    ssim_mean, ssim_lo, ssim_hi = bootstrap_ci(ssim_vals)
    psnr_mean, psnr_lo, psnr_hi = bootstrap_ci(psnr_vals)

    print(f"SSIM: {ssim_mean:.4f} [{ssim_lo:.4f}, {ssim_hi:.4f}]")
    print(f"PSNR: {psnr_mean:.2f} [{psnr_lo:.2f}, {psnr_hi:.2f}]")

    # normalize for ReID
    recon_np = recon.squeeze(1).numpy()
    recon_np = (recon_np - recon_np.min()) / (recon_np.max() - recon_np.min() + 1e-8)

    for arch in ["resnet18", "densenet121"]:
        model = load_reid_model(arch)

        ds = AdaptiveNormDataset(
            recon_np,
            labels,
            recon_np.mean(),
            recon_np.std()
        )
        dl = DataLoader(ds, batch_size=32)

        correct1, correct5 = [], []

        with torch.no_grad():
            for xb, yb in dl:
                logits = model(xb.to(DEVICE))
                probs = torch.softmax(logits, dim=1)

                preds = logits.argmax(1).cpu().numpy()
                yb = yb.numpy()

                correct1.extend((preds == yb).astype(int))

                p_np = probs.cpu().numpy()
                for i in range(len(yb)):
                    top5 = np.argsort(p_np[i])[-5:]
                    correct5.append(int(yb[i] in top5))

        correct1 = np.array(correct1)
        correct5 = np.array(correct5)

        t1_mean, t1_lo, t1_hi = bootstrap_ci(correct1)
        t5_mean, t5_lo, t5_hi = bootstrap_ci(correct5)

        print(f"{arch}: Top1={t1_mean:.4f}, Top5={t5_mean:.4f}")

        rows.append({
            "dataset": "OASIS",
            "method": "U-Net",
            "model": MODEL_DISPLAY[arch],
            "condition": condition,
            "ssim_mean": ssim_mean,
            "ssim_ci_lower": ssim_lo,
            "ssim_ci_upper": ssim_hi,
            "psnr_mean": psnr_mean,
            "psnr_ci_lower": psnr_lo,
            "psnr_ci_upper": psnr_hi,
            "rank1_mean": t1_mean,
            "rank1_ci_lower": t1_lo,
            "rank1_ci_upper": t1_hi,
            "rank5_mean": t5_mean,
            "rank5_ci_lower": t5_lo,
            "rank5_ci_upper": t5_hi,
        })

        del model
        torch.cuda.empty_cache()
        gc.collect()

df = pd.DataFrame(rows)
save_path = SAVE_DIR / "oasis_unet_recon_ci.csv"
df.to_csv(save_path, index=False)

print("\n✓ OASIS U-Net CI DONE")
print("Saved to:", save_path)
display(df)

Loaded OASIS:
  raw: torch.Size([207, 1, 224, 224])
  labels: (207,)

=== GS0 ===
SSIM: 0.9372 [0.9323, 0.9418]
PSNR: 33.44 [32.94, 33.92]
✓ Loaded resnet18
resnet18: Top1=0.2415, Top5=0.5459
✓ Loaded densenet121
densenet121: Top1=0.4589, Top5=0.7295

=== GS50 ===
SSIM: 0.7186 [0.7127, 0.7241]
PSNR: 26.50 [26.29, 26.70]
✓ Loaded resnet18
resnet18: Top1=0.0290, Top5=0.0918
✓ Loaded densenet121
densenet121: Top1=0.0193, Top5=0.0725

✓ OASIS U-Net CI DONE
Saved to: /home/jupyter/notebooks/reid_clean/full_pipeline/results/oasis/recon/oasis_unet_recon_ci.csv


,dataset,method,model,condition,ssim_mean,ssim_ci_lower,ssim_ci_upper,psnr_mean,psnr_ci_lower,psnr_ci_upper,rank1_mean,rank1_ci_lower,rank1_ci_upper,rank5_mean,rank5_ci_lower,rank5_ci_upper
0,OASIS,U-Net,ResNet-18,gs0,0.937204,0.932338,0.941810,33.439915,32.937874,33.923180,0.241546,0.183575,0.304348,0.545894,0.473430,0.613527
1,OASIS,U-Net,DenseNet-121,gs0,0.937204,0.932338,0.941810,33.439915,32.937874,33.923180,0.458937,0.391304,0.526570,0.729469,0.671498,0.787440
2,OASIS,U-Net,ResNet-18,gs50,0.718589,0.712727,0.724056,26.497198,26.292843,26.699631,0.028986,0.009662,0.053140,0.091787,0.053140,0.130556
3,OASIS,U-Net,DenseNet-121,gs50,0.718589,0.712727,0.724056,26.497198,26.292843,26.699631,0.019324,0.004831,0.038647,0.072464,0.043357,0.111111


In [2]:
# ══════════════════════════════════════════════
# OASIS DPS SSIM + PSNR CI 
# ══════════════════════════════════════════════

import numpy as np
import pandas as pd
from PIL import Image
from glob import glob
import os

# ───────────────────────────────────────────────────────────────────────
# CONFIG
# ───────────────────────────────────────────────────────────────────────
BASE = "/home/jupyter/notebooks/reid_clean"
SAVE_PATH = f"{BASE}/full_pipeline/results/oasis/recon/oasis_dps_intensity_ci.csv"

BOOTSTRAP_B = 2000
CI_LOW = 2.5
CI_HIGH = 97.5

DPS_BASE = "/home/jupyter/notebooks/Diffusion Posterior Sampling/diffusion-posterior-sampling/results"

# ───────────────────────────────────────────────────────────────────────
# LOAD PNG PAIRS (GUARANTEED ALIGNMENT)
# ───────────────────────────────────────────────────────────────────────
def load_pairs(orig_dir, recon_dir):
    orig_files  = sorted(glob(os.path.join(orig_dir, "*.png")))
    recon_files = sorted(glob(os.path.join(recon_dir, "*.png")))

    assert len(orig_files) == len(recon_files), "Mismatch in DPS files"

    raw, recon = [], []

    for f1, f2 in zip(orig_files, recon_files):
        raw.append(np.array(Image.open(f1).convert("L")))
        recon.append(np.array(Image.open(f2).convert("L")))

    raw   = np.stack(raw).astype(np.float32) / 255.0
    recon = np.stack(recon).astype(np.float32) / 255.0

    return raw, recon

# ───────────────────────────────────────────────────────────────────────
# METRICS
# ───────────────────────────────────────────────────────────────────────
def psnr(x, y):
    mse = ((x - y) ** 2).mean(axis=(1,2))
    return 10 * np.log10(1.0 / (mse + 1e-8))

def ssim_simple(x, y):
    # stable, fast, per-image SSIM
    mu_x = x.mean(axis=(1,2))
    mu_y = y.mean(axis=(1,2))

    var_x = x.var(axis=(1,2))
    var_y = y.var(axis=(1,2))
    cov   = ((x - mu_x[:,None,None]) * (y - mu_y[:,None,None])).mean(axis=(1,2))

    C1 = 0.01**2
    C2 = 0.03**2

    return ((2*mu_x*mu_y + C1)*(2*cov + C2)) / \
           ((mu_x**2 + mu_y**2 + C1)*(var_x + var_y + C2))

# ───────────────────────────────────────────────────────────────────────
# BOOTSTRAP
# ───────────────────────────────────────────────────────────────────────
def bootstrap_ci(vals):
    vals = np.array(vals)
    rng = np.random.default_rng(42)

    samples = []
    for _ in range(BOOTSTRAP_B):
        idx = rng.choice(len(vals), len(vals), replace=True)
        samples.append(vals[idx].mean())

    samples = np.array(samples)

    return (
        vals.mean(),
        np.percentile(samples, CI_LOW),
        np.percentile(samples, CI_HIGH)
    )

# ───────────────────────────────────────────────────────────────────────
# RUN
# ───────────────────────────────────────────────────────────────────────
results = []

for tag, label in [
    ("OASIS_gs0p_v2",  "GS0"),
    ("OASIS_gs50p_v2", "GS50"),
]:

    print(f"\n=== DPS {label} ===")

    orig_dir  = os.path.join(DPS_BASE, "OASIS_bench_v2/noise/label")
    recon_dir = os.path.join(DPS_BASE, f"{tag}/noise/recon")

    raw, recon = load_pairs(orig_dir, recon_dir)

    print("Loaded:", raw.shape)

    ssim_vals = ssim_simple(recon, raw)
    psnr_vals = psnr(recon, raw)

    ssim_mean, ssim_lo, ssim_hi = bootstrap_ci(ssim_vals)
    psnr_mean, psnr_lo, psnr_hi = bootstrap_ci(psnr_vals)

    print(f"SSIM: {ssim_mean:.4f} [{ssim_lo:.4f}, {ssim_hi:.4f}]")
    print(f"PSNR: {psnr_mean:.2f} [{psnr_lo:.2f}, {psnr_hi:.2f}]")

    results.append({
        "dataset": "OASIS",
        "method": "DPS",
        "condition": label,
        "ssim_mean": ssim_mean,
        "ssim_ci_lower": ssim_lo,
        "ssim_ci_upper": ssim_hi,
        "psnr_mean": psnr_mean,
        "psnr_ci_lower": psnr_lo,
        "psnr_ci_upper": psnr_hi,
        "n_samples": len(raw)
    })

df = pd.DataFrame(results)
df.to_csv(SAVE_PATH, index=False)

print("\n✓ DPS INTENSITY CI DONE")
print("Saved to:", SAVE_PATH)
display(df)


=== DPS GS0 ===
Loaded: (50, 256, 256)
SSIM: 0.3990 [0.3814, 0.4198]
PSNR: 5.94 [5.80, 6.11]

=== DPS GS50 ===
Loaded: (50, 256, 256)
SSIM: 0.7095 [0.6936, 0.7254]
PSNR: 12.09 [11.72, 12.43]

✓ DPS INTENSITY CI DONE
Saved to: /home/jupyter/notebooks/reid_clean/full_pipeline/results/oasis/recon/oasis_dps_intensity_ci.csv


,dataset,method,condition,ssim_mean,ssim_ci_lower,ssim_ci_upper,psnr_mean,psnr_ci_lower,psnr_ci_upper,n_samples
0,OASIS,DPS,GS0,0.399022,0.381425,0.419822,5.939314,5.796964,6.112054,50
1,OASIS,DPS,GS50,0.709529,0.693568,0.725353,12.091235,11.718878,12.427916,50


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# OASIS DPS RE-ID CI 
# ═══════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

# ───────────────────────────────────────────────────────────────────────
# CONFIG
# ───────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_PATH = "/home/jupyter/notebooks/reid_clean/full_pipeline/results/oasis/recon/oasis_dps_reid_ci.csv"

REID_MODELS_ROOT = "/home/jupyter/notebooks/reid_clean/full_pipeline/models/oasis/reid"

# 🔴 CORRECT LABEL SOURCE (from your DPS pipeline)
BENCH_LABEL_PATH = "/home/jupyter/notebooks/Diffusion Posterior Sampling/diffusion-posterior-sampling/data/brain_gs/reid_labels.npy"

BOOTSTRAP_B = 2000
CI_LOW = 2.5
CI_HIGH = 97.5

# ───────────────────────────────────────────────────────────────────────
# LOAD LABELS (NO GUESSING)
# ───────────────────────────────────────────────────────────────────────
bench_labels = np.load(BENCH_LABEL_PATH)
print("Loaded labels:", bench_labels.shape)

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
import os

def load_recons(recon_dir):
    """Load reconstructed images from DPS output directory in correct order."""
    # recon files are named 00000.png, 00001.png etc.
    recon_path = os.path.join(recon_dir, 'noise', 'recon')
    files = sorted(glob(os.path.join(recon_path, '*.png')))
    
    print(f"Found {len(files)} files in {recon_path}")
    
    images = []
    for f in files:
        img = np.array(Image.open(f).convert('L'))  # grayscale
        images.append(img)
    
    arr = np.stack(images, axis=0)
    print(f"Array shape: {arr.shape}, dtype: {arr.dtype}")
    print(f"min: {arr.min()}, max: {arr.max()}")
    return arr

base = '/home/jupyter/notebooks/Diffusion Posterior Sampling/diffusion-posterior-sampling/results'

bench_recons = load_recons(os.path.join(base, 'OASIS_bench_v2'))
gs0p_recons  = load_recons(os.path.join(base, 'OASIS_gs0p_v2'))
gs50p_recons = load_recons(os.path.join(base, 'OASIS_gs50p_v2'))

# ============================================================
# ReID EVALUATION — BENCH + DIFFUSION RECONSTRUCTIONS
# Full pipeline from scratch
# ============================================================

import gc
import pickle
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

REID_MODELS_ROOT = Path("/home/jupyter/notebooks/reid_clean/full_pipeline/models/oasis/reid")
REID_DATA_ROOT   = Path("/home/jupyter/notebooks/reid_clean/full_pipeline/data/oasis/processed/reid")
TEST_PKL_PATH    = Path("/home/jupyter/notebooks/jan26-LearningT/re_id_oracle/"
                        "oasis_data_multi_sample/test.pkl")
BENCH_LABEL_PATH = Path("/home/jupyter/notebooks/Diffusion Posterior Sampling/"
                        "diffusion-posterior-sampling/data/brain_gs/reid_labels.npy")

TARGET_RES = 248   # oracle training resolution

# ------------------------------------------------------------
# 2) BIT-PERFECT UTILITIES
# ------------------------------------------------------------
def enforce_zero_one(data: np.ndarray) -> np.ndarray:
    d_min, d_max = data.min(), data.max()
    return (data - d_min) / (d_max - d_min + 1e-8)

def calculate_intra_class_position(labels: np.ndarray) -> np.ndarray:
    positions = np.zeros_like(labels)
    counts = {}
    for i, lbl in enumerate(labels):
        counts[lbl] = counts.get(lbl, 0) + 1
        positions[i] = counts[lbl]
    return positions

def resize_to_248(imgs_uint8: np.ndarray) -> np.ndarray:
    """Resize (N, H, W) uint8 array to (N, 248, 248) using INTER_AREA."""
    out = np.stack([
        cv2.resize(img, (TARGET_RES, TARGET_RES), interpolation=cv2.INTER_AREA)
        for img in imgs_uint8
    ])
    return out

class AdaptiveNormDataset(Dataset):
    def __init__(self, imgs, lbls, mean, std):
        self.imgs = imgs
        self.lbls = lbls
        self.mean = mean
        self.std  = std
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = self.imgs[idx].copy()
        img = (img - self.mean) / (self.std + 1e-8)
        return (
            torch.tensor(img).unsqueeze(0).float(),
            torch.tensor(self.lbls[idx], dtype=torch.long),
        )

def get_model_architecture(model_name: str, num_classes: int):
    if model_name == "resnet18":
        model = models.resnet18(weights=None)
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(512, num_classes))
    else:
        model = models.densenet121(weights=None)
        model.features[0] = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.classifier = nn.Sequential(nn.Dropout(0.0), nn.Linear(1024, num_classes))
    return model.to(DEVICE)

# ------------------------------------------------------------
# 3) LOAD LABELS
# ------------------------------------------------------------
print("[1/4] Loading labels...")

bench_labels = np.load(BENCH_LABEL_PATH)   # (50,) — same order for all conditions
positions    = calculate_intra_class_position(bench_labels)
max_pos      = int(positions.max())

print(f"  Bench labels shape     : {bench_labels.shape}")
print(f"  Unique subjects        : {len(np.unique(bench_labels))}")
print(f"  Max intra-class pos    : {max_pos}  "
      f"(subjects with 2 slices: "
      f"{[int(l) for l in np.unique(bench_labels) if (bench_labels==l).sum()>1]})")

# ------------------------------------------------------------
# 4) LOAD POOL STATS (train+val at 248x248 — oracle normalization)
# ------------------------------------------------------------
print("\n[2/4] Computing pool normalization stats (train+val)...")

with open(REID_DATA_ROOT / "train.pkl", "rb") as f:
    tr_d = pickle.load(f)
with open(REID_DATA_ROOT / "val.pkl", "rb") as f:
    va_d = pickle.load(f)

pool_imgs  = enforce_zero_one(
    np.array([x["image"] for x in tr_d] + [x["image"] for x in va_d])
)
POOL_MEAN = pool_imgs.mean()
POOL_STD  = pool_imgs.std()
print(f"  Pool Mean : {POOL_MEAN:.4f}")
print(f"  Pool Std  : {POOL_STD:.4f}")

del pool_imgs, tr_d, va_d
gc.collect()

# ------------------------------------------------------------
# 5) PREPARE ALL CONDITIONS
# ------------------------------------------------------------
print("\n[3/4] Preparing all evaluation conditions...")

# ── Condition 1: bench_raw — original 248x248 from test.pkl ──────────────────
test_data = pickle.load(open(TEST_PKL_PATH, "rb"))
images_all = [item['image'] for item in test_data]

np.random.seed(42)
indices = np.random.choice(len(images_all), size=50, replace=False)
indices = np.sort(indices)

bench_raw_imgs = enforce_zero_one(
    np.stack([images_all[i] for i in indices])
)   # (50, 248, 248) float32
print(f"  bench_raw   shape : {bench_raw_imgs.shape}  "
      f"mean={bench_raw_imgs.mean():.4f}")

del test_data, images_all
gc.collect()

# ── Conditions 2-4: diffusion outputs — resize 256→248, enforce_zero_one ─────
# Assumes bench_recons, gs0p_recons, gs50p_recons are already in scope
# as (50, 256, 256) uint8 numpy arrays

def prepare_diffusion(arr_uint8: np.ndarray, name: str) -> np.ndarray:
    assert arr_uint8.dtype == np.uint8, f"{name} must be uint8"
    assert arr_uint8.shape == (50, 256, 256), f"{name} unexpected shape"
    resized = resize_to_248(arr_uint8)                  # (50, 248, 248) uint8
    normed  = enforce_zero_one(resized.astype(np.float32))
    print(f"  {name:15s} shape : {normed.shape}  mean={normed.mean():.4f}")
    return normed

bench_recons_imgs  = prepare_diffusion(bench_recons,  "bench_recons")
gs0p_recons_imgs   = prepare_diffusion(gs0p_recons,   "gs0p_recons")
gs50p_recons_imgs  = prepare_diffusion(gs50p_recons,  "gs50p_recons")

# ── All conditions registry ───────────────────────────────────────────────────
conditions = [
    ("bench_raw",     bench_raw_imgs,    POOL_MEAN, POOL_STD),
    ("bench_recons",  bench_recons_imgs, POOL_MEAN, POOL_STD),
    ("gs0p_recons",   gs0p_recons_imgs,  POOL_MEAN, POOL_STD),
    ("gs50p_recons",  gs50p_recons_imgs, POOL_MEAN, POOL_STD),
]

# ───────────────────────────────────────────────────────────────────────
# DATASET (MATCHES YOUR PIPELINE)
# ───────────────────────────────────────────────────────────────────────
class AdaptiveNormDataset(Dataset):
    def __init__(self, imgs, lbls, mean, std):
        self.imgs = imgs
        self.lbls = lbls
        self.mean = mean
        self.std  = std

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].copy()
        img = (img - self.mean) / (self.std + 1e-8)
        return (
            torch.tensor(img).unsqueeze(0).float(),
            torch.tensor(self.lbls[idx], dtype=torch.long),
        )

# ───────────────────────────────────────────────────────────────────────
# MODEL LOADER (MATCHES YOUR PIPELINE)
# ───────────────────────────────────────────────────────────────────────
def get_model(model_name):
    if model_name == "resnet18":
        model = models.resnet18(weights=None)
        model.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        model.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(512,347))
    else:
        model = models.densenet121(weights=None)
        model.features[0] = nn.Conv2d(1,64,7,2,3,bias=False)
        model.classifier = nn.Sequential(nn.Dropout(0.0), nn.Linear(1024,347))
    return model.to(DEVICE)

# ───────────────────────────────────────────────────────────────────────
# BOOTSTRAP
# ───────────────────────────────────────────────────────────────────────
def bootstrap_ci(vals):
    vals = np.array(vals)
    rng = np.random.default_rng(42)

    samples = []
    for _ in range(BOOTSTRAP_B):
        idx = rng.choice(len(vals), len(vals), replace=True)
        samples.append(vals[idx].mean())

    samples = np.array(samples)

    return (
        vals.mean(),
        np.percentile(samples, CI_LOW),
        np.percentile(samples, CI_HIGH)
    )



# ───────────────────────────────────────────────────────────────────────
# RUN FUNCTION
# ───────────────────────────────────────────────────────────────────────
def evaluate_condition(name, imgs, mean, std):

    results = []

    print(f"\n=== {name} ===")

    for model_name, fname in [
        ("resnet18", "resnet18_Pretrained_Raw_best.pth"),
        ("densenet121", "densenet121_Pretrained_Raw_best.pth"),
    ]:

        model = get_model(model_name)

        ckpt = torch.load(
            os.path.join(REID_MODELS_ROOT, model_name, fname),
            map_location=DEVICE,
            weights_only=False
        )

        sd = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
        model.load_state_dict({k.replace("module.", ""): v for k,v in sd.items()})
        model.eval()

        dl = DataLoader(
            AdaptiveNormDataset(imgs, bench_labels, mean, std),
            batch_size=32,
            shuffle=False
        )

        logits_all = []

        with torch.no_grad():
            for xb, _ in dl:
                logits_all.append(model(xb.to(DEVICE)).cpu())

        logits = torch.cat(logits_all)

        preds = logits.argmax(dim=1).numpy()

        top1 = (preds == bench_labels).astype(float)

        top5_idx = logits.topk(5, dim=1).indices.numpy()
        top5 = np.array([bench_labels[i] in top5_idx[i] for i in range(len(bench_labels))], dtype=float)

        t1_mean, t1_lo, t1_hi = bootstrap_ci(top1)
        t5_mean, t5_lo, t5_hi = bootstrap_ci(top5)

        print(f"{model_name} Top1: {t1_mean:.4f} [{t1_lo:.4f}, {t1_hi:.4f}]")
        print(f"{model_name} Top5: {t5_mean:.4f} [{t5_lo:.4f}, {t5_hi:.4f}]")

        results.append({
            "dataset": "OASIS",
            "method": "DPS",
            "condition": name,
            "model": "ResNet-18" if model_name=="resnet18" else "DenseNet-121",
            "rank1_mean": t1_mean,
            "rank1_ci_lower": t1_lo,
            "rank1_ci_upper": t1_hi,
            "rank5_mean": t5_mean,
            "rank5_ci_lower": t5_lo,
            "rank5_ci_upper": t5_hi,
        })

    return results

# ───────────────────────────────────────────────────────────────────────
# INPUTS (YOU ALREADY LOADED THESE)
# ───────────────────────────────────────────────────────────────────────
# REQUIRED VARIABLES:
# bench_recons_imgs
# gs0p_recons_imgs
# gs50p_recons_imgs
# POOL_MEAN, POOL_STD

all_results = []

all_results += evaluate_condition("RAW", bench_recons_imgs, POOL_MEAN, POOL_STD)
all_results += evaluate_condition("GS0", gs0p_recons_imgs, POOL_MEAN, POOL_STD)
all_results += evaluate_condition("GS50", gs50p_recons_imgs, POOL_MEAN, POOL_STD)

df = pd.DataFrame(all_results)
df.to_csv(SAVE_PATH, index=False)

print("\n✓ DPS RE-ID CI COMPLETE")
print("Saved to:", SAVE_PATH)
display(df)

Loaded labels: (50,)
Found 50 files in /home/jupyter/notebooks/Diffusion Posterior Sampling/diffusion-posterior-sampling/results/OASIS_bench_v2/noise/recon
Array shape: (50, 256, 256), dtype: uint8
min: 0, max: 255
Found 50 files in /home/jupyter/notebooks/Diffusion Posterior Sampling/diffusion-posterior-sampling/results/OASIS_gs0p_v2/noise/recon
Array shape: (50, 256, 256), dtype: uint8
min: 0, max: 255
Found 50 files in /home/jupyter/notebooks/Diffusion Posterior Sampling/diffusion-posterior-sampling/results/OASIS_gs50p_v2/noise/recon
Array shape: (50, 256, 256), dtype: uint8
min: 1, max: 255
[1/4] Loading labels...
  Bench labels shape     : (50,)
  Unique subjects        : 48
  Max intra-class pos    : 2  (subjects with 2 slices: [149, 210])

[2/4] Computing pool normalization stats (train+val)...
  Pool Mean : 0.1867
  Pool Std  : 0.1742

[3/4] Preparing all evaluation conditions...
  bench_raw   shape : (50, 248, 248)  mean=0.2013
  bench_recons    shape : (50, 248, 248)  mean=0.

,dataset,method,condition,model,rank1_mean,rank1_ci_lower,rank1_ci_upper,rank5_mean,rank5_ci_lower,rank5_ci_upper
0,OASIS,DPS,RAW,ResNet-18,0.28,0.16,0.40,0.42,0.28,0.56
1,OASIS,DPS,RAW,DenseNet-121,0.24,0.12,0.36,0.40,0.26,0.54
2,OASIS,DPS,GS0,ResNet-18,0.02,0.00,0.06,0.06,0.00,0.14
3,OASIS,DPS,GS0,DenseNet-121,0.00,0.00,0.00,0.04,0.00,0.10
4,OASIS,DPS,GS50,ResNet-18,0.02,0.00,0.06,0.08,0.02,0.16
5,OASIS,DPS,GS50,DenseNet-121,0.00,0.00,0.00,0.02,0.00,0.06


## Oracle Blind

In [13]:
import torch
import torch.nn as nn
import torchvision.models as tv_models
from torch.utils.data import DataLoader, TensorDataset
import pickle
import numpy as np
from scipy import stats
import pandas as pd
from collections import Counter
from pathlib import Path
import os

BASE     = "/home/jupyter/notebooks/reid_clean"
SAVE_DIR = Path(f"{BASE}/full_pipeline/results/oasis/recon")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
B_BOOT, SEED = 2000, 42
print("Device:", DEVICE)

OASIS_REID_MODEL_DIR = f"{BASE}/full_pipeline/models/oasis/reid"
OASIS_TEST_PKL       = f"{BASE}/full_pipeline/data/oasis/processed/reid/test.pkl"

GS_LEVELS_OASIS = {
    'raw' :'Raw',  'gs0' :'GS_0',  'gs10':'GS_10',
    'gs20':'GS_20','gs30':'GS_30', 'gs40':'GS_40','gs50':'GS_50',
}

# ── Model builder — index 0 = Flatten (no params), index 1 = Linear ───────
def build_oasis_reid_model(arch, gs_str, n_classes):
    fname     = f"{arch}_Pretrained_{gs_str}_best.pth"
    ckpt_path = f"{OASIS_REID_MODEL_DIR}/{arch}/{fname}"

    if arch == 'resnet18':
        m = tv_models.resnet18(weights=None)
        m.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        in_feat = m.fc.in_features
        m.fc    = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_feat, n_classes)
        )
    else:
        m = tv_models.densenet121(weights=None)
        m.features.conv0 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        in_feat = m.classifier.in_features
        m.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_feat, n_classes)
        )

    state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    elif isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    m.load_state_dict(state, strict=True)
    m.eval()
    return m.to(DEVICE), ckpt_path

# ── Inference ──────────────────────────────────────────────────────────────
def get_predictions(model, tensor, batch_size=256):
    ds = TensorDataset(tensor)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for (xb,) in dl:
            preds.append(model(xb.to(DEVICE)).argmax(dim=1).cpu())
    return torch.cat(preds).numpy()

# ── Subject-level Top-1 ────────────────────────────────────────────────────
def subject_top1(preds, labels, subject_ids):
    unique_subj = np.unique(subject_ids)
    correct = 0
    for s in unique_subj:
        idx        = np.where(subject_ids == s)[0]
        true_label = labels[idx[0]]
        vote       = Counter(preds[idx]).most_common(1)[0][0]
        if vote == true_label:
            correct += 1
    return correct / len(unique_subj)

# ── Bootstrap ─────────────────────────────────────────────────────────────
def bootstrap_top1(preds, labels, subject_ids, B=2000, seed=42):
    rng         = np.random.default_rng(seed)
    unique_subj = np.unique(subject_ids)
    point       = subject_top1(preds, labels, subject_ids)
    boot_accs   = []
    for _ in range(B):
        samp = rng.choice(unique_subj, size=len(unique_subj), replace=True)
        idx  = np.concatenate([np.where(subject_ids==s)[0] for s in samp])
        boot_accs.append(
            subject_top1(preds[idx], labels[idx], subject_ids[idx]))
    boot_accs = np.array(boot_accs)
    return (point*100,
            np.percentile(boot_accs, 2.5)*100,
            np.percentile(boot_accs, 97.5)*100)

# ── Load OASIS raw test data ───────────────────────────────────────────────
with open(OASIS_TEST_PKL, 'rb') as f:
    oasis_test = pickle.load(f)

oasis_images   = np.stack([e['image']      for e in oasis_test]).astype(np.float32)
oasis_labels   = np.array([e['reid_label'] for e in oasis_test])
oasis_subjects = np.array([e['subject_id'] for e in oasis_test])

pool_mean    = oasis_images.mean()
pool_std     = oasis_images.std()
oasis_norm   = (oasis_images - pool_mean) / (pool_std + 1e-8)
oasis_tensor = torch.tensor(oasis_norm[:, None, :, :])

N_CLASSES = len(np.unique(oasis_labels))
print(f"OASIS: {oasis_tensor.shape}, "
      f"subjects={len(np.unique(oasis_subjects))}, "
      f"n_classes={N_CLASSES}")

# ── Run ────────────────────────────────────────────────────────────────────
results = []
ARCHS   = ['resnet18', 'densenet121']

for arch in ARCHS:
    arch_label = 'ResNet-18' if arch == 'resnet18' else 'DenseNet-121'
    print(f"\n{'='*55}")
    print(f"OASIS | {arch_label} | Blind (trained GS → test raw)")
    print(f"{'='*55}")
    for cond_key, gs_str in GS_LEVELS_OASIS.items():
        try:
            model, ckpt = build_oasis_reid_model(arch, gs_str, N_CLASSES)
            preds       = get_predictions(model, oasis_tensor)
            pt, lo, hi  = bootstrap_top1(
                preds, oasis_labels, oasis_subjects, B_BOOT, SEED)
            print(f"  {cond_key:6s}: {pt:.2f}% [{lo:.2f}, {hi:.2f}]  "
                  f"← {ckpt.split('/')[-1]}")
            results.append({
                'dataset'  : 'OASIS',
                'model'    : arch_label,
                'condition': cond_key,
                'setting'  : 'blind',
                'mean'     : round(pt, 4),
                'ci_lo'    : round(lo, 4),
                'ci_hi'    : round(hi, 4),
            })
            del model
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"  {cond_key:6s}: ERROR — {e}")

# ── Save ───────────────────────────────────────────────────────────────────
blind_df = pd.DataFrame(results)
out_path = f"{BASE}/CI_results/oasis_blind_oracle_ci.csv"
os.makedirs(f"{BASE}/CI_results", exist_ok=True)
blind_df.to_csv(out_path, index=False)
print("\n=== OASIS BLIND ORACLE CI ===")
print(blind_df.to_string(index=False))
print(f"\nSaved: {out_path}")

Device: cuda
OASIS: torch.Size([1041, 1, 248, 248]), subjects=347, n_classes=347

OASIS | ResNet-18 | Blind (trained GS → test raw)
  raw   : 90.78% [88.64, 93.24]  ← resnet18_Pretrained_Raw_best.pth
  gs0   : 27.67% [23.98, 31.31]  ← resnet18_Pretrained_GS_0_best.pth
  gs10  : 0.58% [0.00, 0.95]  ← resnet18_Pretrained_GS_10_best.pth
  gs20  : 0.86% [0.00, 1.42]  ← resnet18_Pretrained_GS_20_best.pth
  gs30  : 0.29% [0.00, 0.48]  ← resnet18_Pretrained_GS_30_best.pth
  gs40  : 0.29% [0.00, 0.48]  ← resnet18_Pretrained_GS_40_best.pth
  gs50  : 0.29% [0.00, 0.48]  ← resnet18_Pretrained_GS_50_best.pth

OASIS | DenseNet-121 | Blind (trained GS → test raw)
  raw   : 93.66% [91.78, 95.75]  ← densenet121_Pretrained_Raw_best.pth
  gs0   : 28.24% [24.54, 31.84]  ← densenet121_Pretrained_GS_0_best.pth
  gs10  : 0.00% [0.00, 0.00]  ← densenet121_Pretrained_GS_10_best.pth
  gs20  : 0.29% [0.00, 0.48]  ← densenet121_Pretrained_GS_20_best.pth
  gs30  : 0.58% [0.00, 0.95]  ← densenet121_Pretrained_GS_3